# 모델 고도화 과정 — 실험 차트
5가지 실험의 실측 결과를 정리합니다. 모든 수치는 실제 OOF/리더보드 측정값입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('figures', exist_ok=True)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

COLOR_BEFORE = '#B0B8C1'
COLOR_AFTER_GOOD = '#2E86C1'
COLOR_AFTER_BAD = '#C0392B'
COLOR_ACCEPT = '#27AE60'
COLOR_REJECT = '#95A5A6'

## 1. 타겟 분산 분해 — 절대좌표에 숨은 자명한 정보

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
categories = ['절대좌표\n(end_x)', '상대좌표\n(dx)']
between_var = [315.39, 36.39]   # 시작 위치(zone)로 설명되는 분산
within_var = [253.60, 223.40]   # zone 내부 잔차 분산 (실제 예측 대상)

ax.bar(categories, between_var, color='#B0463D', edgecolor='black', linewidth=0.8,
       width=0.5, label='시작 위치(zone)로 설명되는 분산')
ax.bar(categories, within_var, bottom=between_var, color='#2E86C1', edgecolor='black',
       linewidth=0.8, width=0.5, label='zone 내부 잔차 분산 (실제 예측 대상)')

ax.set_ylabel('타겟 분산 (variance)', fontsize=11)
ax.set_title('타겟 분산 분해: 절대좌표 vs 이동량(dx)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9.5, loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.savefig('figures/card1_relative_coords.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 2. 과적합 위험 피처 재검토 — KS 검정과 실측 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# OOF
ax1 = axes[0]
labels = ['피처 유지\n(V7 style)', '피처 제거\n(V8 style)']
values = [14.33, 14.51]
colors = [COLOR_ACCEPT, COLOR_REJECT]
ax1.bar(labels, values, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
ax1.set_ylabel('OOF 점수 (m)', fontsize=11)
ax1.set_title('OOF 비교', fontsize=12, fontweight='bold')
ax1.set_ylim(14.0, 14.7)

# Leaderboard
ax2 = axes[1]
values2 = [14.58, 14.73]
ax2.bar(labels, values2, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
ax2.set_ylabel('리더보드 점수', fontsize=11)
ax2.set_title('리더보드 비교', fontsize=12, fontweight='bold')
ax2.set_ylim(14.0, 15.0)

fig.suptitle('과적합 위험 피처 유지 vs 제거', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/card2_feature_removal.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 3. Y축 대칭 데이터 증강 (기각)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

ax1 = axes[0]
labels = ['증강 없음', 'Y축 증강']
values = [14.3015, 14.2840]
colors = [COLOR_REJECT, COLOR_REJECT]
bars = ax1.bar(labels, values, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for bar, val in zip(bars, values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{val:.4f}',
             ha='center', fontsize=11, fontweight='bold')
ax1.set_ylabel('전체 OOF', fontsize=11)
ax1.set_title('전체 OOF — 거의 무변화', fontsize=12, fontweight='bold')
ax1.set_ylim(14.2, 14.35)

ax2 = axes[1]
values2 = [13.0231, 13.0344]
bars2 = ax2.bar(labels, values2, color=[COLOR_ACCEPT, COLOR_REJECT], edgecolor='black', linewidth=0.8, width=0.5)
for bar, val in zip(bars2, values2):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f'{val:.4f}',
             ha='center', fontsize=11, fontweight='bold')
ax2.set_ylabel('dy RMSE', fontsize=11)
ax2.set_title('dy RMSE — 오히려 미세 악화', fontsize=12, fontweight='bold')
ax2.set_ylim(13.0, 13.06)

fig.suptitle('Y축 대칭 데이터 증강 — 기각 (트리 모델에는 효과 없음)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/card3_yflip_augmentation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 4. 앙상블 가중치 최적화 (기각)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1 = axes[0]
labels = ['균등 가중치\n(1/3, 1/3, 1/3)', '최적화 가중치\n(LGB.36 XGB.21 CAT.43)']
values = [14.3254, 14.3240]
bars = ax1.bar(labels, values, color=[COLOR_REJECT, COLOR_REJECT], edgecolor='black', linewidth=0.8, width=0.5)
for bar, val in zip(bars, values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.0008, f'{val:.4f}',
             ha='center', fontsize=11, fontweight='bold')
ax1.set_ylabel('OOF', fontsize=11)
ax1.set_title('OOF 비교 — 개선폭 0.0014m (노이즈 수준)', fontsize=11, fontweight='bold')
ax1.set_ylim(14.32, 14.33)

ax2 = axes[1]
model_names = ['LightGBM', 'XGBoost', 'CatBoost']
opt_weights = [0.36, 0.214, 0.426]
equal_weights = [1/3, 1/3, 1/3]
x = np.arange(3)
w = 0.35
ax2.bar(x-w/2, equal_weights, w, label='균등', color=COLOR_BEFORE, edgecolor='black', linewidth=0.7)
ax2.bar(x+w/2, opt_weights, w, label='최적화', color='#F39C12', edgecolor='black', linewidth=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(model_names, fontsize=10)
ax2.set_ylabel('앙상블 가중치', fontsize=11)
ax2.set_title('모델별 가중치 비교', fontsize=11, fontweight='bold')
ax2.legend(fontsize=10)

fig.suptitle('LGB/XGB/CatBoost 앙상블 가중치 최적화 — 기각', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/card4_ensemble_weights.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 5. 미사용 피처 발굴 — result_name의 독립 신호 검증

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# zone-controlled effect
ax0 = axes[0]
zones = ['수비', '중원', '공격']
success_dist = [22.23, 20.80, 13.53]
fail_dist = [29.88, 24.25, 12.76]
x = np.arange(3)
w = 0.35
ax0.bar(x-w/2, success_dist, w, label='성공', color='#2E86C1', edgecolor='black', linewidth=0.7)
ax0.bar(x+w/2, fail_dist, w, label='실패', color='#C0392B', edgecolor='black', linewidth=0.7)
ax0.set_xticks(x)
ax0.set_xticklabels(zones, fontsize=11)
ax0.set_ylabel('평균 패스 거리 (m)', fontsize=11)
ax0.set_title('존별 성공/실패 패스 거리', fontsize=11, fontweight='bold')
ax0.legend(fontsize=10)

# OOF
ax1 = axes[1]
labels = ['베이스라인\n(미사용)', 'result_name\n추가']
values = [14.33, 14.07]
ax1.bar(labels, values, color=[COLOR_BEFORE, COLOR_ACCEPT], edgecolor='black', linewidth=0.8, width=0.5)
ax1.set_ylabel('OOF 점수 (m)', fontsize=11)
ax1.set_title('OOF 비교', fontsize=12, fontweight='bold')
ax1.set_ylim(13.8, 14.5)

# Leaderboard
ax2 = axes[2]
values2 = [14.58, 14.16]
ax2.bar(labels, values2, color=[COLOR_BEFORE, COLOR_ACCEPT], edgecolor='black', linewidth=0.8, width=0.5)
ax2.set_ylabel('리더보드 점수', fontsize=11)
ax2.set_title('리더보드 비교', fontsize=12, fontweight='bold')
ax2.set_ylim(13.8, 14.8)

fig.suptitle('미사용 피처 발굴: result_name(패스 성공 여부)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/card5_result_feature.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 종합: 5가지 실험 전체 비교

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

experiments = ['1. 상대좌표\n전환', '2. 피처\n제거', '3. Y축\n증강', '4. 앙상블\n가중치', '5. result_name\n추가']
improvements = [20.29-14.50, 14.58-14.73, 0.0, 14.3254-14.3240, 14.58-14.16]
decisions = ['채택', '기각', '기각', '기각', '채택']
colors_summary = [COLOR_ACCEPT if d=='채택' else COLOR_REJECT for d in decisions]

bars = ax.bar(experiments, improvements, color=colors_summary, edgecolor='black', linewidth=0.8)
ax.axhline(0, color='black', linewidth=1)
for bar, val, dec in zip(bars, improvements, decisions):
    y = bar.get_height()
    va = 'bottom' if y >= 0 else 'top'
    offset = 0.1 if y >= 0 else -0.1
    ax.text(bar.get_x()+bar.get_width()/2, y+offset, f'{dec}\n({val:+.2f})',
            ha='center', va=va, fontsize=10, fontweight='bold')

ax.set_ylabel('점수 개선폭 (양수 = 개선)', fontsize=11)
ax.set_title('5가지 고도화 시도 — 개선폭 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/card_summary_all.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()